# Inverted Pendulum -- Control Algorithms (Solution)

This is the **solution** notebook for the Inverted Pendulum simulation
(`simulation.py`), which uses the [`InvertedPendulum-v5`](https://gymnasium.farama.org/environments/mujoco/inverted_pendulum/)
environment from Gymnasium.

**Environment recap**

| | shape | meaning |
|---|---|---|
| Observation | `[x, theta, x_dot, theta_dot]` | cart position, pole angle (rad, 0 = upright), cart velocity, pole angular velocity |
| Action | `Box(-3.0, 3.0, (1,))` | force applied to the cart (N) |
| Reward | +1 per step the pole stays upright | episode ends if the pole tips too far or the cart goes out of bounds |

Each section below explains one of the three control strategies used by the
simulator and shows a complete, working implementation. Run the **Export**
cell at the bottom to write these implementations to `../controllers.py` (the
file used by `simulation.py`), then run:

```bash
python simulation.py
```

from the project root to see them in action.


In [ ]:
import os

import numpy as np


## 1. Manual Control

The simplest "controller" is you! In `simulation.py`, on every simulation
step the GUI checks which arrow keys are currently held down and passes that
information to `get_manual_action` as a dictionary, e.g.:

```python
pressed_keys = {"left": True, "right": False, "reset": False, "quit": False}
```

Your job is to turn this into a force command for the cart:

- If `"left"` is held, push the cart in the negative direction.
- If `"right"` is held, push the cart in the positive direction.
- If both or neither are held, apply zero net force.
- The result must be a NumPy array of shape `(1,)`, clipped to the bounds of
  `action_space` (`action_space.low` / `action_space.high`).

**Implementation steps**
1. Default `magnitude` to `action_space.high[0]` if not provided.
2. Start with `force = 0.0`.
3. Subtract `magnitude` if `pressed_keys["left"]` is true; add `magnitude` if
   `pressed_keys["right"]` is true.
4. Wrap `force` in a NumPy array with dtype `action_space.dtype`.
5. Clip to `[action_space.low, action_space.high]` with `np.clip` and return.


In [ ]:
def get_manual_action(action_space, pressed_keys, magnitude=None):
    """
    Convert currently-held keyboard keys into an action.

    Parameters
    ----------
    action_space : gymnasium.spaces.Box
        The environment's action space.
    pressed_keys : dict
        Maps key names ("left", "right") to booleans indicating whether the
        corresponding arrow key is currently held down.
    magnitude : float, optional
        Magnitude of the force to apply. Defaults to the maximum force
        allowed by `action_space`.

    Returns
    -------
    numpy.ndarray
        The action to send to env.step, clipped to action_space.
    """
    if magnitude is None:
        magnitude = float(action_space.high[0])

    force = 0.0
    if pressed_keys.get("left"):
        force -= magnitude
    if pressed_keys.get("right"):
        force += magnitude

    action = np.array([force], dtype=action_space.dtype)
    return np.clip(action, action_space.low, action_space.high)


## 2. PID Control

A **PID controller** computes a control signal from the *error* between a
desired setpoint and the current measurement:

$$ u(t) = K_p \, e(t) \;+\; K_i \int_0^t e(\tau)\, d\tau \;+\; K_d \frac{de(t)}{dt} $$

- **P (proportional)** -- reacts to the *current* error. Larger `Kp` means a
  stronger immediate correction, but too large causes oscillation.
- **I (integral)** -- accumulates *past* error over time, eliminating steady
  state bias. Too large causes overshoot/oscillation ("integral windup").
- **D (derivative)** -- reacts to *how fast* the error is changing, damping
  oscillations. Too large amplifies sensor noise.

For the inverted pendulum, the quantity we want to drive to zero is the pole
angle `theta = observation[1]` (0 = perfectly upright), so:

```python
error = theta - setpoint   # setpoint is usually 0
```

The simulation calls `compute_action` once per simulation step (every `dt`
seconds, where `dt = env.unwrapped.dt`), and calls `set_gains` whenever the
user moves a slider.

**Implementation steps for `compute_action`**
1. Compute `error = theta - self.setpoint` where `theta = observation[1]`.
2. Update the running integral: `self._integral += error * dt`.
3. Compute the derivative using a finite difference:
   `derivative = (error - self._prev_error) / dt`.
4. Save `self._prev_error = error` for the next call.
5. Combine the terms:
   `output = self.kp * error + self.ki * self._integral + self.kd * derivative`.
6. Wrap `output` in a NumPy array (dtype `action_space.dtype`) and clip to
   `[action_space.low, action_space.high]`.

**Implementation steps for `reset`**
- Reset `self._integral` and `self._prev_error` back to `0.0`. This is called
  at the start of every episode so old errors don't leak into the new one.

**Try it out:** once exported, start the simulator's PID mode and try
`Kp=20, Ki=0, Kd=2` as a starting point -- then use the sliders to explore how
each term changes the behaviour.


In [ ]:
class PIDController:
    """
    A PID controller that balances the pole at `setpoint` (default: upright,
    theta = 0) by applying a force to the cart:

        u(t) = Kp * e(t) + Ki * integral(e) + Kd * d(e)/dt

    where e(t) = theta(t) - setpoint.
    """

    def __init__(self, kp=0.0, ki=0.0, kd=0.0, setpoint=0.0):
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.setpoint = setpoint
        self.reset()

    def reset(self):
        """Clear the integral and derivative memory (call between episodes)."""
        self._integral = 0.0
        self._prev_error = 0.0

    def set_gains(self, kp, ki, kd):
        """Update the P, I and D gains (e.g. from GUI sliders)."""
        self.kp = kp
        self.ki = ki
        self.kd = kd

    def compute_action(self, observation, action_space, dt):
        """
        Compute the control action for the current observation.

        Parameters
        ----------
        observation : array-like
            [x, theta, x_dot, theta_dot] from the environment.
        action_space : gymnasium.spaces.Box
            The environment's action space, used to clip the output.
        dt : float
            Time step between calls (env.unwrapped.dt).

        Returns
        -------
        numpy.ndarray
            The action to send to env.step, clipped to action_space.
        """
        theta = observation[1]
        error = theta - self.setpoint

        self._integral += error * dt
        derivative = (error - self._prev_error) / dt if dt > 0 else 0.0
        self._prev_error = error

        output = self.kp * error + self.ki * self._integral + self.kd * derivative

        action = np.array([output], dtype=action_space.dtype)
        return np.clip(action, action_space.low, action_space.high)


## 3. Reinforcement Learning Control (PPO via Stable-Baselines3)

[Stable-Baselines3](https://stable-baselines3.readthedocs.io/) provides
ready-made implementations of popular RL algorithms. We'll use **PPO**
(Proximal Policy Optimization), a policy-gradient algorithm that works well
on continuous-control tasks like this one.

At a high level, PPO repeatedly:
1. Runs the current policy in the environment to collect a batch of
   `(state, action, reward)` experience (`n_steps` steps).
2. Computes advantage estimates (how much better an action was than average),
   controlled by `gamma` (discount factor) and `gae_lambda`.
3. Updates the policy network with several epochs of mini-batch gradient
   descent (`batch_size`), nudged by `learning_rate`, while *clipping* the
   policy update so it doesn't change too drastically (the "proximal" part).
4. `ent_coef` adds an entropy bonus to encourage exploration.

This repeats until `total_timesteps` environment steps have been collected.
The simulation's GUI lets the user edit all of these hyperparameters before
training starts.

**Implementation steps for `train_rl_agent`**
1. Import `PPO` from `stable_baselines3`.
2. Construct `PPO("MlpPolicy", env, ...)`, passing through the hyperparameters
   from the `hyperparams` dict using `.get(key, default)` (so missing keys
   fall back to sensible defaults), plus `verbose=1`.
3. Call `model.learn(total_timesteps=total_timesteps, callback=callback)`.
4. Make sure the directory containing `save_path` exists
   (`os.makedirs(..., exist_ok=True)`), then call `model.save(save_path)`.
5. Return the trained model.

**Implementation steps for `load_rl_agent`**
1. Import `PPO` from `stable_baselines3`.
2. Return `PPO.load(path, env=env)`.

**Implementation steps for `get_rl_action`**
1. Call `model.predict(observation, deterministic=deterministic)`, which
   returns `(action, state)`.
2. Return just the `action`.


In [ ]:
def train_rl_agent(env, hyperparams, total_timesteps, save_path, callback=None):
    """
    Train a PPO agent on `env` and save it to `save_path`.

    Parameters
    ----------
    env : gymnasium.Env
        The (non-rendered) training environment.
    hyperparams : dict
        PPO hyperparameters. Recognised keys: learning_rate, n_steps,
        batch_size, gamma, gae_lambda, ent_coef. Missing keys fall back to
        Stable-Baselines3 defaults.
    total_timesteps : int
        Number of environment steps to train for.
    save_path : str
        Path (without/with .zip) to save the trained model checkpoint.
    callback : stable_baselines3.common.callbacks.BaseCallback, optional
        Callback passed through to model.learn (e.g. for progress reporting
        in the GUI).

    Returns
    -------
    stable_baselines3.PPO
        The trained model.
    """
    from stable_baselines3 import PPO

    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=hyperparams.get("learning_rate", 3e-4),
        n_steps=hyperparams.get("n_steps", 1024),
        batch_size=hyperparams.get("batch_size", 64),
        gamma=hyperparams.get("gamma", 0.99),
        gae_lambda=hyperparams.get("gae_lambda", 0.95),
        ent_coef=hyperparams.get("ent_coef", 0.0),
        verbose=1,
    )

    model.learn(total_timesteps=total_timesteps, callback=callback)

    save_dir = os.path.dirname(os.path.abspath(save_path))
    os.makedirs(save_dir, exist_ok=True)
    model.save(save_path)

    return model


def load_rl_agent(path, env=None):
    """
    Load a previously-trained PPO model from `path`.

    Parameters
    ----------
    path : str
        Path to the saved model (.zip checkpoint).
    env : gymnasium.Env, optional
        Environment to attach to the loaded model.

    Returns
    -------
    stable_baselines3.PPO
        The loaded model.
    """
    from stable_baselines3 import PPO

    return PPO.load(path, env=env)


def get_rl_action(model, observation, deterministic=True):
    """
    Get the action chosen by a trained RL model for `observation`.

    Parameters
    ----------
    model : stable_baselines3.PPO
        A trained (or loaded) model.
    observation : array-like
        [x, theta, x_dot, theta_dot] from the environment.
    deterministic : bool
        Whether to use the deterministic policy (recommended for evaluation).

    Returns
    -------
    numpy.ndarray
        The action to send to env.step.
    """
    action, _ = model.predict(observation, deterministic=deterministic)
    return action


## Export to `controllers.py`

Running the cell below uses `inspect.getsource` to pull the source code of
each function/class defined above and writes it to `../controllers.py`, the
file used by `simulation.py`.

```bash
python simulation.py
```


In [ ]:
import ast
import os

_OUTPUT_PATH = os.path.join("..", "controllers.py")

_HEADER = '''"""
Control algorithms for the Gymnasium InvertedPendulum-v5 environment.

Auto-generated by exporting controllers_solution.ipynb -- re-run the export cell in that
notebook after making changes to regenerate this file.
"""

import os

import numpy as np
'''

# Names of the functions/classes to export, in the order they should appear
# in controllers.py.
_NAMES = ["get_manual_action", "PIDController", "train_rl_agent", "load_rl_agent", "get_rl_action"]


def _find_definition(name):
    """Find the source of the most recently executed `def`/`class name`.

    `inspect.getsource` doesn't work reliably for classes defined in
    notebooks, so instead we search the IPython input history (`In`) for a
    top-level function/class definition called `name` and pull out just that
    definition using the `ast` module.
    """
    for cell_source in reversed(In):
        try:
            tree = ast.parse(cell_source)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                if node.name == name:
                    return ast.get_source_segment(cell_source, node)
    raise RuntimeError(f"Could not find a definition for `{name}`. Did you run that cell?")


with open(_OUTPUT_PATH, "w") as f:
    f.write(_HEADER)
    for _name in _NAMES:
        f.write("\n\n")
        f.write(_find_definition(_name))
        f.write("\n")

print(f"Wrote {os.path.abspath(_OUTPUT_PATH)}")
